Calculating Expected Loss 

- **probability of default (PD) * Loss given default (LGD) * Exposure at default (EAD)**
- establish dependant variable ie what to predict - default or non default
- default definition - eg payment 90 days overdue, fraud
- new variable - 1 default, 0 non default
- Logistic regression where the dependant var is whether the customer defaulted or  not, estimates the relationship between two things.
- Outcome of interest is the non default or default event
- Odds ratio = ratio of non defaults to defaults
- All independent variables should be dummy variables/binary categorical variables or indicator variables
- Transform continous variables to dummy variables
- The greater the value of independent variable the lower the probability of default
- **Fine classing** - turning continous variables into categories
- Weight of evidence (WoE) - shows to what extend an independent variable will predict a dependant variable
   - The formula of the weight of evidence is the natural logarithm of the ratio of the proportion of observations of the first type of outcome of the dependent variable that fall into the respective category of the independent variable and the proportion of observations of the second type of outcome of the dependent variable that fall into the respective category of the independent variable.
   - The two types of outcome are non defaulted or good and defaulted or bad. So the weight of evidence would be the natural logarithm of the ratio of the proportion of goods from the total number of goods that fall into the category to the proportion of bads from the total number of bads that fall into a category.
- **Course classing** - constructing new catgeories based on the intitial ones. Categories with similar WoE are bundled together which lowers the number of dummies hence improving the pd model
- Information value used for var selection, preselect predictors  - how much information the original dependant variable brings with respect to explaining the dependant variable IV < 0.02 no predictive power, 0.02 < IV < 0.1 weak power, 0.1 <IV 0.3 medium power, 01. <IV <0.5 strong, 0,5 < IV suspiciously high too good to be true 

## PD model

### Data preparation

**Dependant variable : Good/bad (default) Definition default and non default accounts**

#### Import libraries and data 

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
# Load the file
loan_data = pd.read_parquet('loan_data_cleaned.parquet')

In [15]:
loan_data.head()

,Unnamed: 0,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,...,addr_state : TX,addr_state : UT,addr_state : VA,addr_state : VT,addr_state : WA,addr_state : WI,addr_state : WV,addr_state : WY,initial_list_status : f,initial_list_status : w
0,0,1077501,1296599,5000,5000,4975.0,36 months,10.65,162.87,B,...,0,0,0,0,0,0,0,0,1,0
1,1,1077430,1314167,2500,2500,2500.0,60 months,15.27,59.83,C,...,0,0,0,0,0,0,0,0,1,0
2,2,1077175,1313524,2400,2400,2400.0,36 months,15.96,84.33,C,...,0,0,0,0,0,0,0,0,1,0
3,3,1076863,1277178,10000,10000,10000.0,36 months,13.49,339.31,C,...,0,0,0,0,0,0,0,0,1,0
4,4,1075358,1311748,3000,3000,3000.0,60 months,12.69,67.79,B,...,0,0,0,0,0,0,0,0,1,0


#### Default and non default flag

In [16]:
#data about borrower perfomance and repayment behaviour
loan_data ['loan_status'].unique()

array(['Fully Paid', 'Charged Off', 'Current', 'Default',
       'Late (31-120 days)', 'In Grace Period', 'Late (16-30 days)',
       'Does not meet the credit policy. Status:Fully Paid',
       'Does not meet the credit policy. Status:Charged Off'],
      dtype=object)

In [17]:
#see how many values are there for each status
loan_data['loan_status'].value_counts()

loan_status
Current                                                224226
Fully Paid                                             184739
Charged Off                                             42475
Late (31-120 days)                                       6900
In Grace Period                                          3146
Does not meet the credit policy. Status:Fully Paid       1988
Late (16-30 days)                                        1218
Default                                                   832
Does not meet the credit policy. Status:Charged Off       761
Name: count, dtype: int64

In [18]:
#see proportion of accounts by status
loan_data ['loan_status'].value_counts() / loan_data['loan_status'].count()

loan_status
Current                                                0.480878
Fully Paid                                             0.396193
Charged Off                                            0.091092
Late (31-120 days)                                     0.014798
In Grace Period                                        0.006747
Does not meet the credit policy. Status:Fully Paid     0.004263
Late (16-30 days)                                      0.002612
Default                                                0.001784
Does not meet the credit policy. Status:Charged Off    0.001632
Name: count, dtype: float64

In [19]:
#store good/bad flag or default 0 /non default 1 indicators
# We create a new variable that has the value of '0' if a condition is met, and the value of '1' if it is not met.
loan_data['good_bad'] = np.where(loan_data['loan_status'].isin(['Charged Off', 'Default',
                                                       'Does not meet the credit policy. Status:Charged Off',
                                                       'Late (31-120 days)']), 0, 1) 

In [20]:
loan_data['good_bad']

0         1
1         0
2         1
3         1
4         1
         ..
466280    1
466281    0
466282    1
466283    1
466284    1
Name: good_bad, Length: 466285, dtype: int64

#### Constructing independent variables

#### Splitting data - solution to overfitting

In [21]:
# Takes a set of inputs and a set of targets as arguments. Splits the inputs and the targets into four dataframes:
# Inputs - Train, Inputs - Test, Targets - Train, Targets - Test
# This is identical to using axis=1 and is harder to mess up:
# loan_data.drop(columns=['good_bad'])
train_test_split(loan_data.drop('good_bad', axis = 1), loan_data['good_bad'])

[        Unnamed: 0        id  member_id  loan_amnt  funded_amnt  \
 460885      460885  10444677   12316824      35000        35000   
 84327        84327   7564869    9247010      12800        12800   
 398266      398266  15390561   17462979       9600         9600   
 261256      261256  33531590   36174866       9000         9000   
 263630      263630  29754062   32277234      24000        24000   
 ...            ...       ...        ...        ...          ...   
 23489        23489    613739     786858      14400        14400   
 183482      183482   1691741    1974395      12250        12250   
 298024      298024  28663907   31197084      21000        21000   
 433842      433842  12457736   14469842      13500        13500   
 392111      392111  16141216   18243707      28000        28000   
 
         funded_amnt_inv        term  int_rate  installment grade  ...  \
 460885          35000.0   36 months     14.47      1204.23     C  ...   
 84327           12800.0   60 mont

In [28]:
loan_data_inputs_train, loan_data_inputs_test,loan_data_targets_train, loan_data_targets_test = train_test_split(loan_data.drop('good_bad', axis = 1), loan_data['good_bad'])

In [29]:
loan_data_inputs_train.shape

(349713, 209)

In [30]:
loan_data_targets_train.shape

(349713,)

In [31]:
loan_data_inputs_test.shape

(116572, 209)

In [34]:
loan_data_targets_test.shape

(116572,)

In [35]:
loan_data_inputs_train, loan_data_inputs_test,loan_data_targets_train, loan_data_targets_test = train_test_split(loan_data.drop('good_bad', axis = 1), loan_data['good_bad'], 
                                                                                                                 test_size = 0.2,
                                                                                                                random_state = 42)

In [37]:
loan_data_inputs_test.shape

(93257, 209)

In [38]:
loan_data_targets_test.shape

(93257,)

#### Data preparation

In [39]:
df_inputs_prepr = loan_data_inputs_train
df_target_prepr = loan_data_targets_train

In [40]:
df_inputs_prepr['grade'].unique()

array(['A', 'C', 'D', 'B', 'E', 'F', 'G'], dtype=object)

In [42]:
df1 = pd.concat([df_inputs_prepr['grade'], df_target_prepr], axis =1)
df1.head()

,grade,good_bad
427211,A,1
206088,C,1
136020,A,1
412305,D,0
36159,C,0
